In [1]:
import requests
import json

# Test the API with a small query
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.cond": "cancer",
    "filter.overallStatus": "COMPLETED",
    "pageSize": 2,
    "format": "json"
}

response = requests.get(BASE_URL, params=params)
print(f"Status code: {response.status_code}")
print(f"URL called: {response.url}")

data = response.json()
print(f"Total studies available: {data.get('totalCount', 'unknown')}")
print(f"Studies returned in this page: {len(data.get('studies', []))}")

Status code: 200
URL called: https://clinicaltrials.gov/api/v2/studies?query.cond=cancer&filter.overallStatus=COMPLETED&pageSize=2&format=json
Total studies available: unknown
Studies returned in this page: 2


In [3]:
params = {
    "query.cond": "cancer",
    "filter.advanced": "AREA[Phase]PHASE2 OR AREA[Phase]PHASE3",
    "pageSize": 2,
    "format": "json",
    "countTotal": "true"
}

response = requests.get(BASE_URL, params=params)
data = response.json()
print(f"Total Phase 2/3 cancer trials: {data.get('totalCount', 'unknown')}")

Total Phase 2/3 cancer trials: 47982


In [2]:
# Look at the structure of a single study
first_study = data['studies'][0]
print(json.dumps(first_study, indent=2)[:5000])  # First 5000 chars to keep it readable

{
  "protocolSection": {
    "identificationModule": {
      "nctId": "NCT02009449",
      "orgStudyIdInfo": {
        "id": "17159"
      },
      "secondaryIdInfos": [
        {
          "id": "J1L-AM-JZGA",
          "type": "OTHER",
          "domain": "Eli Lilly and Company"
        },
        {
          "id": "AM0010-001",
          "type": "OTHER",
          "domain": "ARMO BioSciences"
        }
      ],
      "organization": {
        "fullName": "Eli Lilly and Company",
        "class": "INDUSTRY"
      },
      "briefTitle": "A Phase 1 Study of Pegilodecakin (LY3500518) in Participants With Advanced Solid Tumors",
      "officialTitle": "A Phase 1, Open-Label Dose Escalation First-in-Human Study to Evaluate the Tolerability, Safety, Maximum Tolerated Dose, Preliminary Clinical Activity and Pharmacokinetics of AM0010 in Patients With Advanced Solid Tumors",
      "acronym": "IVY"
    },
    "statusModule": {
      "statusVerifiedDate": "2024-11",
      "overallStatus": "COM

In [4]:
# Extract the fields we need for our schema
study = data['studies'][0]
protocol = study['protocolSection']

# trials table fields
nct_id = protocol['identificationModule'].get('nctId')
brief_title = protocol['identificationModule'].get('briefTitle')
official_title = protocol['identificationModule'].get('officialTitle')
overall_status = protocol['statusModule'].get('overallStatus')
phases = protocol.get('designModule', {}).get('phases', [])
enrollment = protocol.get('designModule', {}).get('enrollmentInfo', {}).get('count')
start_date = protocol.get('statusModule', {}).get('startDateStruct', {}).get('date')

print(f"NCT ID: {nct_id}")
print(f"Brief title: {brief_title}")
print(f"Status: {overall_status}")
print(f"Phases: {phases}")
print(f"Enrollment: {enrollment}")
print(f"Start date: {start_date}")

# sponsors
sponsors_module = protocol.get('sponsorCollaboratorsModule', {})
lead_sponsor = sponsors_module.get('leadSponsor', {})
collaborators = sponsors_module.get('collaborators', [])
print(f"\nLead sponsor: {lead_sponsor.get('name')} (class: {lead_sponsor.get('class')})")
print(f"Collaborators: {[c.get('name') for c in collaborators]}")

# conditions
conditions = protocol.get('conditionsModule', {}).get('conditions', [])
print(f"\nConditions: {conditions}")

# locations
locations = protocol.get('contactsLocationsModule', {}).get('locations', [])
print(f"\nNumber of locations: {len(locations)}")
if locations:
    print(f"First location: {locations[0].get('facility')}, {locations[0].get('city')}, {locations[0].get('country')}")

NCT ID: NCT02677155
Brief title: Sequential Intranodal Immunotherapy (SIIT) Combined With Anti-PD1 (Pembrolizumab) in Follicular Lymphoma
Status: COMPLETED
Phases: ['PHASE2']
Enrollment: 10
Start date: 2016-01

Lead sponsor: Oslo University Hospital (class: OTHER)
Collaborators: ['Norwegian Cancer Society', 'Merck Sharp & Dohme LLC']

Conditions: ['Follicular Lymphoma']

Number of locations: 1
First location: Oslo University Hospital Radiumhospitalet, Oslo, Norway


In [1]:
# Test the parser module on one raw file
import sys
sys.path.append("..")  # so we can import from src/

from src.parsers import parse_trial, parse_sponsors, parse_conditions, parse_locations
import json

# Load one page of data
with open("../data/raw/page_0001.json", "r", encoding="utf-8") as f:
    studies = json.load(f)

print(f"Loaded {len(studies)} studies from page 1")

# Parse the first study and show results
sample = studies[0]
print("\n--- TRIAL ---")
trial = parse_trial(sample)
for k, v in trial.items():
    print(f"  {k}: {v}")

print("\n--- SPONSORS ---")
for s in parse_sponsors(sample):
    print(f"  {s}")

print("\n--- CONDITIONS ---")
print(f"  {parse_conditions(sample)}")

print("\n--- LOCATIONS (first 3) ---")
for loc in parse_locations(sample)[:3]:
    print(f"  {loc}")

Loaded 1000 studies from page 1

--- TRIAL ---
  nct_id: NCT02677155
  brief_title: Sequential Intranodal Immunotherapy (SIIT) Combined With Anti-PD1 (Pembrolizumab) in Follicular Lymphoma
  official_title: Open Label, Phase II, Study: Sequential Intranodal Immunotherapy (SIIT) Combined With Anti-PD1 (Pembrolizumab) in Patients With Stage III/IV Untreated and Relapsed Follicular Lymphoma
  overall_status: COMPLETED
  phase: PHASE2
  study_type: INTERVENTIONAL
  enrollment_count: 10
  start_date: 2016-01-01
  completion_date: 2021-02-01
  primary_completion_date: 2021-02-01
  why_stopped: None
  has_results: False
  primary_intervention_type: RADIATION
  primary_intervention_name: Radiotherapy

--- SPONSORS ---
  {'sponsor_name': 'Oslo University Hospital', 'sponsor_class': 'OTHER', 'lead_or_collaborator': 'LEAD'}
  {'sponsor_name': 'Norwegian Cancer Society', 'sponsor_class': 'OTHER', 'lead_or_collaborator': 'COLLABORATOR'}
  {'sponsor_name': 'Merck Sharp & Dohme LLC', 'sponsor_class':